# RTM3D/KM3D ResNet-18 — full KITTI validation parity

**Revision:** `RTM3D-KM3D-ONNX-FULLVAL-PARITY-2026-10-08-r1`

This notebook runs all 3,769 Chen validation images through the reviewed RTM3D ResNet-18 checkpoint in PyTorch (Colab GPU) and its exported ONNX model (CPU). Both sets of raw heads use the same pinned RTM3D CUDA geometry decoder. The notebook saves KITTI-format predictions, compares decoded boxes and 3D geometry, and computes descriptive AP_R40 with the MobileADAS3D project evaluator.

The 16-image raw-head audit had a small `hps` tolerance miss. That is recorded but does not block this follow-up if all fixture outputs are finite and the fixture run is complete; the point here is to measure whether it changes decoded predictions across the full split. This is not an official KITTI leaderboard result, an iPhone speed test, or deployment qualification. Keep the Colab GPU enabled. The full run can take a while; logs and outputs are saved in Drive.

## 1. Mount Drive, update project code, and check GPU/runtime

Run from the top. This does not replace the Colab PyTorch/CUDA stack.

In [ ]:
from pathlib import Path
from datetime import datetime
import importlib.util, json, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/mobile_adas3d')
PROJECT_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
if not PROJECT_DIR.exists(): subprocess.run(['git','clone','--branch','main','--single-branch',PROJECT_URL,str(PROJECT_DIR)],check=True)
origin = subprocess.check_output(['git','-C',str(PROJECT_DIR),'remote','get-url','origin'],text=True).strip()
if origin.startswith('git@github.com:'): origin='https://github.com/'+origin.split(':',1)[1]
if origin.rstrip('/').removesuffix('.git').lower() != 'https://github.com/ali-rt/mobile_adas3d': raise RuntimeError(f'Unexpected project origin: {origin!r}')
branch = subprocess.check_output(['git','-C',str(PROJECT_DIR),'branch','--show-current'],text=True).strip()
if branch != 'main': raise RuntimeError(f'Expected main, found {branch!r}; preserve this checkout and use a clean main checkout.')
dirty = subprocess.check_output(['git','-C',str(PROJECT_DIR),'status','--porcelain','--untracked-files=no'],text=True).strip()
if dirty: raise RuntimeError('MobileADAS3D has tracked edits; preserve them and use a clean main checkout.')
subprocess.run(['git','-C',str(PROJECT_DIR),'pull','--ff-only','origin','main'],check=True)
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0,str(PROJECT_DIR))
SCRIPT = PROJECT_DIR / 'scripts/audit_rtm3d_km3d_res18_onnx_fullval.py'
if not SCRIPT.is_file(): raise FileNotFoundError(f'{SCRIPT} missing from main; confirm this revision is published.')
missing=[]
if importlib.util.find_spec('onnxruntime') is None: missing.append('onnxruntime')
if importlib.util.find_spec('cv2') is None: missing.append('opencv-python-headless')
if missing: subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
import cv2, onnxruntime as ort, torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun this cell.')
print('MobileADAS3D commit:',subprocess.check_output(['git','-C',str(PROJECT_DIR),'rev-parse','--short','HEAD'],text=True).strip())
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'GPU:',torch.cuda.get_device_name(0))
print('ONNX Runtime:',ort.__version__,'providers:',ort.get_available_providers(),'OpenCV:',cv2.__version__)

## 2. Locate the reviewed bundle, checkpoint, complete split, data, and pinned RTM3D source

This cell preserves old checkouts and outputs. If an expected input is missing, add the correct path here; do not replace or delete existing experiment artifacts.

In [ ]:
from scripts.run_rtm3d_km3d_res18_smoke import RTM3D_COMMIT, RTM3D_URL
EXPORT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/rtm3d_km3d_res18_r1')
CHECKPOINT=EXPORT_ROOT/'model_res18_1.pth'
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
EXPECTED_COMMIT='888c379e79d8a6d134f06a9b7d669118679e06dc'
if RTM3D_COMMIT != EXPECTED_COMMIT: raise RuntimeError(f'Unexpected pinned commit constant: {RTM3D_COMMIT}')
EXPORT_ROOT.mkdir(parents=True,exist_ok=True)
bundles=sorted(EXPORT_ROOT.glob('onnx_export_r1_*/rtm3d_km3d_res18_onnx_export.zip'),key=lambda p:p.stat().st_mtime,reverse=True)
if not bundles: raise FileNotFoundError('No reviewed ONNX bundle under '+str(EXPORT_ROOT/'onnx_export_r1_*')+'; run the ONNX export notebook first.')
BUNDLE=bundles[0]
if not CHECKPOINT.is_file(): raise FileNotFoundError(CHECKPOINT)
SPLIT=SPLIT_DIR/'val.txt'
if not SPLIT.is_file(): raise FileNotFoundError(SPLIT)
SAMPLE_IDS=[x.strip() for x in SPLIT.read_text().splitlines() if x.strip()]
if len(SAMPLE_IDS)!=3769 or len(set(SAMPLE_IDS))!=3769: raise RuntimeError(f'Expected Chen val IDs 3769 unique, got {len(SAMPLE_IDS)}')
DATA_CANDIDATES=[Path('/content/kitti_m67'),Path('/content/kitti_m66'),Path('/content/kitti_m65'),Path('/content/kitti_m64'),Path('/content/kitti_m62'),Path('/content/kitti_m61'),Path('/content/kitti'),Path('/content/monodetr_kitti_a2'),Path('/content/drive/MyDrive/datasets/kitti')]
DATASET_ROOT=None
for candidate in DATA_CANDIDATES:
    image_dirs=[candidate/'training'/'image_2',candidate/'training'/'image_02']
    label_dirs=[candidate/'training'/'label_2',candidate/'training'/'label_02',candidate/'training'/'label']
    if any((folder/f'{SAMPLE_IDS[0]}.png').is_file() and (folder/f'{SAMPLE_IDS[-1]}.png').is_file() for folder in image_dirs) and (candidate/'training'/'calib'/f'{SAMPLE_IDS[0]}.txt').is_file() and any((folder/f'{SAMPLE_IDS[0]}.txt').is_file() and (folder/f'{SAMPLE_IDS[-1]}.txt').is_file() for folder in label_dirs): DATASET_ROOT=candidate; break
if DATASET_ROOT is None: raise FileNotFoundError('No candidate has KITTI val images, labels, and calibration. Add the correct dataset root to DATA_CANDIDATES.')

def inspect_rtm3d(path):
    if not path.exists(): return False
    try:
        head=subprocess.check_output(['git','-C',str(path),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
        origin=subprocess.check_output(['git','-C',str(path),'remote','get-url','origin'],text=True,stderr=subprocess.DEVNULL).strip()
        dirty=subprocess.check_output(['git','-C',str(path),'status','--porcelain','--untracked-files=all'],text=True).strip()
    except subprocess.CalledProcessError: return False
    return head==EXPECTED_COMMIT and origin.rstrip('/').removesuffix('.git')==RTM3D_URL.removesuffix('.git') and not dirty
base=Path('/content/RTM3D_KM3D_R1_20261007_decoder')
RTM3D_REPO=base; suffix=0
while RTM3D_REPO.exists() and not inspect_rtm3d(RTM3D_REPO):
    suffix+=1; RTM3D_REPO=base.with_name(base.name+f'_fresh{suffix}')
if not RTM3D_REPO.exists():
    subprocess.run(['git','clone',RTM3D_URL,str(RTM3D_REPO)],check=True)
    subprocess.run(['git','-C',str(RTM3D_REPO),'checkout','--detach',EXPECTED_COMMIT],check=True)
if not inspect_rtm3d(RTM3D_REPO): raise RuntimeError('Pinned RTM3D checkout did not pass commit/origin/clean-source checks.')
RUN_TAG=datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUTPUT=EXPORT_ROOT/f'fullval_decoded_r1_{RUN_TAG}'
LOG_DIR=EXPORT_ROOT/'colab_logs'; LOG_DIR.mkdir(parents=True,exist_ok=True)
LOG=LOG_DIR/f'fullval_decoded_r1_{RUN_TAG}.log'
print('Bundle:',BUNDLE,'\nCheckpoint:',CHECKPOINT,'\nRTM3D:',RTM3D_REPO,'\nKITTI:',DATASET_ROOT,'\nChen val:',SPLIT,'(3769 images)','\nOutput:',OUTPUT,'\nLog:',LOG)

## 3. Run full-split PyTorch-vs-ONNX inference, decode, prediction export, and AP_R40 comparison

This is the long-running cell. It writes all 3,769 prediction files for both runtimes, compares matched detections, computes project AP_R40, and stores a durable log. It does not modify the checkpoint or train anything.

In [ ]:
command=[sys.executable,'-u',str(SCRIPT),'--bundle',str(BUNDLE),'--repo',str(RTM3D_REPO),'--checkpoint',str(CHECKPOINT),'--dataset-root',str(DATASET_ROOT),'--split-dir',str(SPLIT_DIR),'--output-dir',str(OUTPUT)]
print('+',' '.join(command),flush=True)
with LOG.open('w',encoding='utf-8') as log:
    process=subprocess.Popen(command,cwd=PROJECT_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True); log.write(line); log.flush()
    code=process.wait(); log.write(f'\nEXIT CODE: {code}\n')
if code: raise RuntimeError(f'Full-val parity failed with exit {code}; preserve output and inspect {LOG}')
REPORT=OUTPUT/'fullval_decoded_parity.json'
report=json.loads(REPORT.read_text())
assert report['complete'] and report['evaluated_images']==3769 and report['prediction_files']=={'pytorch':3769,'onnx':3769}
ARTIFACT_ZIP=OUTPUT/'fullval_results.zip'
with zipfile.ZipFile(ARTIFACT_ZIP,'w',compression=zipfile.ZIP_DEFLATED) as archive:
    for path in [REPORT,OUTPUT/'pytorch_kitti_r40_metrics.csv',OUTPUT/'onnx_kitti_r40_metrics.csv',OUTPUT/'kitti_r40_comparison.csv',LOG]: archive.write(path,arcname=path.name)
print('Full Chen-val inference completed. Report:',REPORT)
print('AP comparison:',OUTPUT/'kitti_r40_comparison.csv')
print('Upload this compact report bundle for review:',ARTIFACT_ZIP)
print('Predictions:',OUTPUT/'pytorch_predictions/data','and',OUTPUT/'onnx_predictions/data')

## 4. Review the result

AP_R40 here is useful for checking whether the ONNX path changes predictions relative to PyTorch; it is still the project's evaluator, not the official KITTI leaderboard. Cloud CPU/GPU timings are not iPhone latency. A stable result is the signal to proceed to the target-device benchmark and app integration.

In [ ]:
import pandas as pd
comparison=pd.read_csv(OUTPUT/'kitti_r40_comparison.csv')
display(comparison)
print('Decoded-pair counts:',report['decoded_pairing'])
print('Matched decoded deltas:',json.dumps(report['decoded_matched_pair_delta_distributions'],indent=2))
print('Model-forward-only timing (cloud; not iPhone):',json.dumps(report['latency_ms_per_image_model_forward_only'],indent=2))
print('Full JSON report:',REPORT)
print('Compact results ZIP:',ARTIFACT_ZIP)
print('Durable log:',LOG)